# Rust 106: Native Collections

Chapter 06 of the Rust track, and the one where the language stops being about single values. Chapters 01–05 built up scalars, strings and the ownership rules; this chapter puts those rules to work inside the containers real programs are made of: `Vec`, `HashMap`, `HashSet`, `BTreeMap`, `VecDeque`, plus arrays, slices and tuples. Every container here is a concrete exercise in the ownership model — that is not a slogan, it is the actual design.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `06_native-collections_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, arriving from Python and C++. You know what a dynamic array, a hash map and a doubly-linked queue are; you will find all three here, but with sizes owned explicitly, iteration governed by the borrow rules, and a compiler that refuses the operations that would invalidate them. What is new is not the data structures — it is who owns them and what that implies.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Several sections build one running example across cells, so out-of-order runs will leave bindings undefined.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic or panic message you would get. §12 is the catalogue.
- The prose is terse and the facts are dense. This is a reference you will come back to, not a novel you read once.

## The one idea to internalise

A collection **owns its elements**. Pushing moves (unless the element is `Copy`), so the collection is the single owner and every read is a borrow of it. From that one sentence everything follows: `Vec::push` taking `&mut self`, iteration handing out `&T` by default, the borrow checker refusing a push in the middle of a loop over the same `Vec`, and why `collect()` can build a new container from borrowed pieces. Python's containers are shared blobs managed by the interpreter; Rust's are values on the stack or heap with exactly one owner, and the compiler checks every access against that fact.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. Arrays and Slices: Fixed vs Borrowed](#1-arrays-and-slices-fixed-vs-borrowed)
- [2. `Vec`: The Dynamic Array](#2-vec-the-dynamic-array)
- [3. Tuples: Anonymous Fixed Sequences](#3-tuples-anonymous-fixed-sequences)
- [4. `String` as a Collection](#4-string-as-a-collection)
- [5. `HashMap`: Key-Value Storage](#5-hashmap-key-value-storage)
- [6. `HashSet`: Membership and Algebra](#6-hashset-membership-and-algebra)
- [7. `BTreeMap` and `BTreeSet`: Sorted Alternatives](#7-btreemap-and-btreeset-sorted-alternatives)
- [8. `VecDeque`: A Ring Buffer](#8-vecdeque-a-ring-buffer)
- [9. Iterating Collections](#9-iterating-collections)
- [10. Collection Patterns You Will Actually Write](#10-collection-patterns-you-will-actually-write)
- [11. Ownership Inside Containers](#11-ownership-inside-containers)
- [12. Reading the Compiler: Panics and E-Codes](#12-reading-the-compiler-panics-and-e-codes)
- [13. Performance Notes for Low-Latency Code](#13-performance-notes-for-low-latency-code)
- [14. Mental Model: Length, Capacity, Owner](#14-mental-model-length-capacity-owner)
- [15. Habits and Review](#15-habits-and-review)
- [16. Cheat Sheet](#16-cheat-sheet)

If you are returning to this chapter later, §16 is the one-page summary, §12 is the diagnostics catalogue, and §13 is where the latency notes live.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 4 is still live in cell 30.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — the deliberate compile errors in this chapter are comments, so every cell still runs. `println!` writes to stdout; `eprintln!` and `dbg!` write to stderr.

### A note on imports

`Vec`, `String`, arrays and tuples are always in scope. The map, set and deque types live in `std::collections`, and **each is imported exactly once**, at its first use (§5 onward). In evcxr a `use` stays in effect for the rest of the session, so later sections rely on the earlier import — after a kernel restart, run from the top and everything is back in scope.


In [ ]:
// 0.1 Smoke test: one container, owned, grown by one move at a time.
let mut ticks = Vec::new();
ticks.push(101);
ticks.push(102);
println!("ticks = {ticks:?} (len {})", ticks.len());


## 1. Arrays and Slices: Fixed vs Borrowed

An **array** `[T; N]` is a fixed-size sequence whose length `N` is part of the type — `[i32; 3]` and `[i32; 4]` are two different types. A **slice** `&[T]` is a borrowed view: a pointer plus a length, with no capacity of its own and no ownership of the data it sees. Arrays store; slices look. (This chapter extends chapter 05's slice material from strings to any element type.)

| Type | What it is | Where it lives | Size in the type |
|---|---|---|---|
| `[T; N]` | owned, fixed length | stack (usually) | length in the type |
| `&[T]` | borrowed view (fat pointer) | pointer on the stack | length erased |
| `Vec<T>` | owned, growable | buffer on the heap | only in `Vec<[T; N]>` |

The pattern you will use for the rest of the chapter: arrays and `Vec`s are *storage*; when you pass them to a function, you pass a *slice* so the function works on both.


In [ ]:
// 1.1 Array literals, their types, indexing and mutation.
let numbers: [i32; 3] = [10, 20, 30];
let threes = [0u8; 3];                 // [value; N] repeats a value
let mut flags = [false; 2];
flags[0] = true;
println!("numbers = {numbers:?}, threes = {threes:?}, flags[1] = {}", flags[1]);
println!("the length is in the type: {}", std::any::type_name::<[i32; 3]>());


`[value; N]` requires the value to be `Copy` or a const-expression — it is initialisation, not `N` moves. Array bounds are checked at runtime exactly like `Vec` indexing, and the compiler rejects out-of-range indices it can prove constant: `numbers[3]` on an `[i32; 3]` is a compile error, not a panic.


In [ ]:
// 1.2 The query methods every sequence shares.
let prices = [101.5, 102.0, 100.75, 99.5];
println!("len = {}, is_empty = {}", prices.len(), prices.is_empty());
println!("first = {:?}, last = {:?}", prices.first(), prices.last());
let empty: [f64; 0] = [];
println!("empty first = {:?}", empty.first());   // Option, not a panic


### Slices: the borrowed view

A slice is created from anything with the right layout — an array, a `Vec`, another slice — with `&` or a range. `&prices[..]` is the whole array as a slice; `&prices[1..3]` is elements 1 and 2. The slice does not own the elements: dropping it frees nothing, and mutating through it requires `&mut`.


In [ ]:
// 1.3 Slicing arrays and Vecs with the same syntax.
let arr = [1, 2, 3, 4, 5];
let all: &[i32] = &arr[..];
let middle: &[i32] = &arr[1..4];       // 1, 2, 3 — end is exclusive
let tail: &[i32] = &arr[3..];
println!("all = {all:?}, middle = {middle:?}, tail = {tail:?}");

fn sum(values: &[i32]) -> i32 {        // slices accept arrays AND Vecs
    values.iter().sum()
}
let vec_form = vec![1, 2, 3];
println!("sum(arr) = {}, sum(vec) = {}", sum(&arr), sum(&vec_form));


`&arr` on an array of length 5 coerces to `&[i32]` automatically — unsized coercion at the argument position. That is why `sum(&arr)` and `sum(&vec_form)` both compile against one `&[i32]` parameter: the function sees the same fat pointer either way.

Ranges work on slices exactly as they did on strings in chapter 04 (`..`, `..=`, open-ended). Out-of-range slicing panics at runtime: `&arr[3..6]` on a length-5 slice panics with *range end index 6 out of range*.


In [ ]:
// 1.4 Storage choice: stack array vs heap Vec.
let small = [0i32; 16];                // 64 bytes on the stack, no allocation
let grown = vec![0i32; 16];            // one heap allocation, resizable
println!("arr len {} (type-fixed), vec len {} (runtime-fixed)", small.len(), grown.len());
// [T; N] cannot grow: there is no push. Choosing the array type is
// choosing "this length is a property of the problem".


## 2. `Vec`: The Dynamic Array

`Vec<T>` is the container you will reach for most: an owned, growable sequence whose buffer lives on the heap. A `Vec` value itself is three machine words — pointer, **length** (elements in use), **capacity** (slots allocated) — so moving a `Vec` never copies elements, it moves three words. Chapter 05's move rules apply unchanged: `push` takes ownership of the element; iterating by value consumes the vector.

| Operation | Complexity | Note |
|---|---|---|
| `push` / `pop` | amortised O(1) | reallocates when length meets capacity |
| `insert` / `remove` at index i | O(n) | shifts the tail |
| indexing `v[i]` | O(1) | panics past the end |
| `get(i)` | O(1) | returns `Option`, never panics |
| iteration | O(n) | `iter()` borrows, `into_iter()` consumes |

The amortised analysis hides the reallocation: when length == capacity, the buffer is abandoned and elements move to a bigger one.


In [ ]:
// 2.1 Growth: capacity jumps, elements are moved, addresses change.
let mut v: Vec<i32> = Vec::new();
println!("new: cap = {}", v.capacity());
for i in 0..6 {
    let before = v.as_ptr();
    v.push(i);
    println!("push {i}: len {} cap {} moved={}", v.len(), v.capacity(), v.as_ptr() != before);
}


### Growing on purpose: `with_capacity`

If the final size is known (or roughly known), start there: `Vec::with_capacity(n)` allocates once and never reallocates while length stays under `n`. For latency-sensitive code this is the difference between one allocation and a reallocation cascade — the amortised average hides spikes that the p99 does not.


In [ ]:
// 2.2 with_capacity reserves once; a plain push loop reallocates.
let known: Vec<i32> = Vec::with_capacity(6);
println!("with_capacity(6): cap = {}, len = {}", known.capacity(), known.len());

let mut reserved = Vec::with_capacity(6);
let base = reserved.as_ptr();
for i in 0..6 {
    reserved.push(i);
    println!("push {i}: moved = {}", reserved.as_ptr() != base);
}


### Building a `Vec`

| Syntax | What happens |
|---|---|
| `vec![a, b, c]` | literal macro, capacity == length == 3 |
| `Vec::new()` + `push` | empty start, grow as needed |
| `Vec::with_capacity(n)` | reserve first, then push |
| `(0..n).collect()` | from an iterator, sized source so one allocation |
| `[T; N]::to_vec()` | copy an array to the heap |
| `iter().cloned()` / `.copied()` | from borrowed elements |


In [ ]:
// 2.3 Five ways to build the same vector.
let a = vec![10, 20, 30];
let mut b = Vec::new();
b.push(10); b.push(20); b.push(30);
let c: Vec<i32> = (10..=30).step_by(10).collect();
let d = [10, 20, 30].to_vec();
let e: Vec<i32> = [10, 20, 30].iter().copied().collect();
println!("{a:?} {b:?} {c:?} {d:?} {e:?}");


### Indexing vs `get`, and the `Option` boundary

`v[i]` panics past the end; `v.get(i)` returns `Option<&T>` and never panics. The rule from chapter 05 returns: index when out-of-range is a bug (let it panic), `get` when absence is expected data. `v[i]` on a `Vec<bool>` yields `bool` because the element is owned by the `Vec` and `bool: Copy` — the general rule is that indexing returns a place (a position you can assign through with a `&mut` borrow), not a copy that pops out.


In [ ]:
// 2.4 Indexing panics, get returns Option, and indexing yields a place.
let mut v = vec![10, 20, 30];
v[0] = 99;                              // indexing as an lvalue: place semantics
println!("v = {v:?}, v[0] = {}, get(1) = {:?}", v[0], v.get(1));
let maybe: Option<&i32> = v.get(9);
println!("get(9) = {maybe:?}");
// let boom = v[9];
// thread 'main' panicked: index out of bounds: the len is 3 but the index is 9


### Removing elements

`pop` takes from the end and returns `Option<T>` (owning the element back). `remove(i)` shifts everything after `i` and returns the element; `drain(range)` removes a run and yields the elements as an iterator; `retain(pred)` keeps what matches and never reallocates. `clear` empties but keeps the buffer — capacity is unchanged, so refilling costs no reallocation.


In [ ]:
// 2.5 Removal inventory.
let mut v = vec![10, 20, 30, 40, 50];
let popped = v.pop();                   // Some(50)
let removed = v.remove(1);              // 20, tail shifts left
let drained: Vec<i32> = v.drain(1..3).collect();  // removes 30, 40
println!("popped={popped:?} removed={removed} drained={drained:?} left={v:?}");
v.retain(|x| x % 10 == 0);              // 10 % 10 == 0: kept
println!("after retain: {v:?}");


### Pushing borrows, popping owns

`v.push(x)` needs `&mut v`. A borrow that **ends before** the push is fine — that is NLL, chapter 05 §4 — but a borrow still live at the push (a held reference used afterwards, or an in-flight iteration over the same `Vec`) is the E0502 story, restated as a container fact. Conversely, `pop`, `remove` and `drain` hand ownership of elements back out. The full worked example of the mutate-while-iterating error is in §11.


In [ ]:
// 2.6 NLL in container terms: the borrow ends, then the push is free.
let mut v = vec![1, 2, 3];
let first: &i32 = &v[0];
println!("first = {first}");            // last use of the borrow
v.push(4);                              // legal: the borrow already ended
println!("after push: {v:?}");
// Using the borrow AFTER the push is the error:
// let held: &i32 = &v[0];
// v.push(5);
// println!("{held}");
// error[E0502]: cannot borrow `v` as mutable because it is also borrowed as immutable


### `Vec<T>` derefs to `&[T]`

Method resolution and deref coercion conspire so that `Vec` behaves like a slice everywhere a slice is wanted: `v.len()`, `v.iter()`, `v.first()` all come from the slice, and a `&Vec<T>` coerces to `&[T]` at any argument position. The signature rule from chapter 05 gets its punchline: **take `&[T]`, not `&Vec<T>`** — the `Vec` caller pays nothing, and arrays join the party for free.


In [ ]:
// 2.7 Vec derefs to slice: one signature, many callers.
fn longest(values: &[i32]) -> Option<usize> {
    values.iter().enumerate().max_by_key(|(_, v)| *v).map(|(i, _)| i)
}
let data = vec![3, 9, 4];
let arr = [7, 2];
println!("vec: {:?}, arr: {:?}",
    longest(&data),
    longest(&arr));                     // &Vec<i32> and &[i32; 2] both coerce


## 3. Tuples: Anonymous Fixed Sequences

A **tuple** is a fixed-length, heterogeneous sequence: `(f64, &str, u32)`. Like an array its length is in the type, but unlike an array the element types may differ, and the elements are addressed by position (`.0`, `.1`, …) rather than by name. Tuples are for *temporary grouping* — a function returning several values, a destructured iterator item — not for data with meaning attached to it. When a tuple starts needing comments to explain `.2`, it wants to be a struct (chapter 10).

| Access | Syntax | Works on |
|---|---|---|
| by index | `t.0`, `t.1`, … `.11` | named positions only, `.0` cannot loop over |
| by pattern | `let (a, b) = t;` | destructuring, including nested patterns |
| as a whole | `t == other`, `t < other` | lexicographic if all elements support it |
| length | always fixed | `.len()` on a tuple is an error |


In [ ]:
// 3.1 Construction, indexed access, and destructuring.
let tick = (101.25, "AAPL", 300u32);
let (price, symbol, size) = tick;      // destructuring copy
println!("price = {price}, symbol = {symbol}, size = {size}");
println!("by index: {} {} {}", tick.0, tick.1, tick.2);


In [ ]:
// 3.2 A tuple in a return position: three values, one move.
fn min_max_sum(values: &[i32]) -> (i32, i32, i32) {
    (values.iter().min().copied().unwrap_or(0),
     values.iter().max().copied().unwrap_or(0),
     values.iter().sum())
}
let (low, high, total) = min_max_sum(&[5, 2, 8, 1, 9]);
println!("low = {low}, high = {high}, total = {total}");


In [ ]:
// 3.3 Comparisons are lexicographic; the unit type is the empty tuple.
println!("(1, 9) < (2, 0) is {}", (1, 9) < (2, 0));
let unit = ();
println!("the unit type prints as {unit:?}");


## 4. `String` as a Collection

`String` has been in every chapter since chapter 01, but it belongs in this one too: it **is** a collection of bytes — literally a `Vec<u8>` wearing a UTF-8 validity guarantee. That framing explains every string fact that has surprised anyone so far: no `s[i]` indexing (bytes are not characters), `len()` counting bytes, `chars()` existing as the explicit character iterator, and slicing panicking when a range splits a code point.

| View | Type | `"héllo"` gives |
|---|---|---|
| bytes | `&[u8]` via `as_bytes()` | 6 |
| chars | `Chars` via `chars()` | 5 |
| whole text | `&str` | the borrowed view over all of it |


In [ ]:
// 4.1 The three views of one text.
let text = String::from("héllo");
println!("bytes: {}", text.as_bytes().len());
println!("chars: {}", text.chars().count());
println!("as Vec<u8>: {}", text.clone().into_bytes().len());


In [ ]:
// 4.2 Char-level mutation, and what it costs.
let mut text = String::from("ab");
text.push('c');                        // amortised O(1), like Vec::push
text.push_str("def");                  // copies a byte run in, may allocate
text.insert(0, 'z');                   // O(n): shifts every byte after it
println!("{text:?} popped = {:?}", text.pop());


Inserting into the middle of a `String` is O(n) for the same reason `Vec::insert` is: every byte after the point shifts. The UTF-8 guarantee narrows the API in exchange: `insert` accepts only `char`, `insert_str` only `&str`, and both re-check validity — there is no way to splice a broken byte sequence into a `String`. That guarantee is also why chapter 04's slicing rule exists: every mutation and every slice must land on a `char` boundary.

Two collection habits carry over unchanged from `Vec`: `pop` returns `Option<char>` (owning the element back), and growing reuses the buffer until capacity runs out. The difference is entirely the validity invariant on top.


## 5. `HashMap`: Key-Value Storage

`HashMap<K, V>` maps keys to values by hashing: the key's `Hash` output picks a bucket, and equality (`Eq`) settles collisions. Both bounds are requirements — a key type must implement `Eq` **and** `Hash`, which is why `f64` is not a valid key (no `Eq`) and why `String` works while `&str` needs a borrowed-form lookup. There is no `HashMap` literal: you build one with `new` + `insert`, `from`, or `collect` from pairs.

| Operation | Signature (roughly) | Returns |
|---|---|---|
| `get(&k)` | `Option<&V>` | never panics |
| `get_mut(&k)` | `Option<&mut V>` | in-place update |
| `insert(k, v)` | `Option<V>` | the **old** value, if the key existed |
| `remove(&k)` | `Option<V>` | the removed pair's value, owned |
| `contains_key(&k)` | `bool` | cheap membership |
| `entry(k)` | `Entry` | the upsert pattern, next cell |

**Iteration order is unspecified — and deliberately unstable.** The hasher is seeded randomly per process, so two identically built maps can iterate in different orders, even in the same program run. If order matters, that is `BTreeMap`'s job (§7).


In [ ]:
// 5.1 Three constructions, one lookup.
use std::collections::HashMap;

let mut a: HashMap<&str, i32> = HashMap::new();
a.insert("AAPL", 101);
a.insert("MSFT", 250);

let b = HashMap::from([("AAPL", 101), ("MSFT", 250)]);
let c: HashMap<&str, i32> = [("AAPL", 101), ("MSFT", 250)].into();

println!("a = {a:?}");
println!("all three agree: {}", a == b && b == c);
println!("get AAPL: {:?}, get TSLA: {:?}", a.get("AAPL"), a.get("TSLA"));


### The `entry` API: lookup-or-insert in one step

Counting, grouping and default-initialising all have the same shape: *get the value if it exists, otherwise install a default, then mutate it*. Python spells this `setdefault` or `defaultdict`; Rust spells it `entry`. `map.entry(key)` returns an `Entry` enum that borrows the map mutably; `or_insert(default)` turns it into a `&mut V`, so the increment happens in place with no second lookup. `or_default()` uses the type's `Default`, and `and_modify` chains an edit onto either case.


In [ ]:
// 5.2 Word counting with entry — the canonical upsert.
let raw = "alpha beta alpha gamma beta alpha";
let mut counts: HashMap<&str, u32> = HashMap::new();
for word in raw.split_whitespace() {
    *counts.entry(word).or_insert(0) += 1;
}
println!("counts = {counts:?}");

let mut caps: HashMap<&str, u32> = HashMap::new();
caps.entry("AAPL").or_insert(50);
caps.entry("AAPL").and_modify(|v| *v += 10).or_insert(0);
println!("caps = {caps:?}");


### The lookup inventory

`get` hands back a shared borrow of the value; `get_mut` a mutable one; `remove` needs `&mut self` and returns the value **by value** — the map stops owning it. All three take the key as `&Q` where `K: Borrow<Q>`, which is the technical reason a `HashMap<String, V>` accepts `get("AAPL")` with a plain `&str`: the map never needs to own the key you are merely asking about.


In [ ]:
// 5.3 get, get_mut, contains_key, remove — and what each returns.
let mut book: HashMap<&str, f64> = HashMap::from([("AAPL", 101.5), ("MSFT", 250.0)]);
println!("contains TSLA: {}", book.contains_key("TSLA"));

if let Some(price) = book.get_mut("AAPL") {
    *price += 0.25;                     // in place, no reinsert
}
println!("after get_mut: {book:?}");

let taken = book.remove("MSFT");        // owned value comes back
println!("removed = {taken:?}, book now = {book:?}");


### Owned keys: `String` in, `&str` out

A `HashMap<String, V>` **owns** its keys — `insert` moves them in, and the caller's binding is dead afterwards, exactly as §2's push-move rule predicts. Lookups still take `&str`. Removing returns the key and value as an owned pair, so data can travel through a map without cloning.


In [ ]:
// 5.4 String keys are moved in; remove hands them back.
let mut sizes: HashMap<String, usize> = HashMap::new();
let symbol = String::from("AAPL");
sizes.insert(symbol, 4);                // moves the String; `symbol` is now dead
println!("lookup by &str: {:?}", sizes.get("AAPL"));

let removed = sizes.remove("AAPL");     // Some((String, usize))
println!("removed = {removed:?}, map = {sizes:?}");
// println!("{symbol}");
// error[E0382]: borrow of moved value: `symbol` -- value moved into `sizes`


## 6. `HashSet`: Membership and Algebra

`HashSet<T>` is `HashMap<T, ()>` with the values hidden: a bag of unique elements, answered by hashing. `insert` returns `bool` — `true` only when the element was new — which makes insert-into-set a one-call duplicate check. As a key requirement, `T` must implement `Eq` + `Hash`. Iteration order: unspecified, exactly as §5 said.

Python's `set` analogue is close enough for intuition, with one mechanical difference: the algebra methods below take the other set **by reference** and return *lazy iterators*, not new sets. Materialising is your call — `.collect()` into whatever you want.

| Method | Yields | Also |
|---|---|---|
| `union(&b)` | elements in either | lazy |
| `intersection(&b)` | elements in both | lazy |
| `difference(&b)` | in `a`, not in `b` | asymmetric |
| `symmetric_difference(&b)` | in exactly one | lazy |
| `is_subset` / `is_superset` / `is_disjoint` | `bool` | eager |


In [ ]:
// 6.1 Membership: insert's bool, contains, remove, len.
use std::collections::HashSet;

let mut watch: HashSet<&str> = HashSet::new();
println!("insert AAPL -> {} (new)", watch.insert("AAPL"));
println!("insert AAPL -> {} (duplicate)", watch.insert("AAPL"));
println!("contains MSFT: {}", watch.contains("MSFT"));
let evicted = watch.remove("AAPL");     // bool, not Option -- nothing to own
println!("remove -> {evicted}, len = {}", watch.len());


In [ ]:
// 6.2 Set algebra on two watchlists.
let a: HashSet<i32> = [100, 200, 300].into_iter().collect();
let b: HashSet<i32> = [300, 400].into_iter().collect();
println!("union        = {:?}", a.union(&b).collect::<Vec<_>>());
println!("intersection = {:?}", a.intersection(&b).collect::<Vec<_>>());
println!("difference   = {:?}", a.difference(&b).collect::<Vec<_>>());
println!("symmetric    = {:?}", a.symmetric_difference(&b).collect::<Vec<_>>());
println!("b subset of a: {}, a disjoint b: {}", b.is_subset(&a), a.is_disjoint(&b));


### No indexing, by design

A set exposes no `s[i]`: an order that changes run to run must not be addressable. You iterate it, query it, or drain it. When a *sorted* view is needed for display, collect into a `Vec` and sort it — or question whether the type you wanted was `BTreeSet` all along, which is the next section's subject.


## 7. `BTreeMap` and `BTreeSet`: Sorted Alternatives

Swap the hash table for a **B-tree** and everything ordering-related falls out for free: keys only need `Ord`, iteration walks in key order, and whole *ranges* can be answered in O(log n + m) without touching unrelated keys. Lookups are O(log n) rather than O(1) — for most workloads the constant hides, and what you gain is determinism: identical insertions always iterate identically, in every run.

| Need | Reach for |
|---|---|
| fastest single-key lookup | `HashMap` |
| sorted iteration, `min`/`max`, range scans | `BTreeMap`/`BTreeSet` |
| "give me keys 10..=20" | `BTreeMap::range` |
| deterministic order across runs | BTree anything |


In [ ]:
// 7.1 Ordered iteration, first/last, and range queries.
use std::collections::BTreeMap;

let prices: BTreeMap<i32, &str> = [(105, "AAPL"), (99, "MSFT"), (102, "TSLA"), (250, "NVDA")].into();
println!("in key order: {prices:?}");
println!("first = {:?}, last = {:?}", prices.first_key_value(), prices.last_key_value());
println!("range 100..=110: {:?}", prices.range(100..=110).collect::<Vec<_>>());


`range` accepts any bound shape: `..=x`, `x..`, exclusive or inclusive — the same range grammar as slices (§1). `BTreeSet` mirrors its map sibling element-wise, and adds ordered pops: `pop_first` / `pop_last` remove and return the extreme element, which is the whole of a priority queue for the price of a `BTreeSet`.


In [ ]:
// 7.2 BTreeSet: sorted iteration, ranges, ordered pops.
use std::collections::BTreeSet;

let mut levels: BTreeSet<u32> = [300, 100, 250, 99].into_iter().collect();
println!("sorted: {levels:?}");
println!("range 100..300: {:?}", levels.range(100..300).collect::<Vec<_>>());
println!("popped first = {:?}, last = {:?}", levels.pop_first(), levels.pop_last());
println!("left = {levels:?}");


## 8. `VecDeque`: A Ring Buffer

`VecDeque<T>` is a growable ring buffer: a fixed allocation viewed as a circle, with **O(1) push and pop at both ends**. A `Vec` can push at the back cheaply but pays O(n) to touch the front (`insert(0, x)` shifts everything); a `VecDeque` pays nothing at either end, at the cost of memory that is not one contiguous run — which is why `as_slices` exists and why slicing a deque is not a thing.

The name says the use cases: a **deque** (double-ended queue) is a sliding window, a work queue, a BFS frontier — anything that feeds at one end and drains at the other.


In [ ]:
// 8.1 Both ends, O(1) each.
use std::collections::VecDeque;

let mut q: VecDeque<i32> = VecDeque::new();
q.push_back(2);
q.push_front(1);
q.push_back(3);
println!("q = {q:?}, front = {:?}, back = {:?}", q.front(), q.back());
let first = q.pop_front();
let last = q.pop_back();
println!("pop_front = {first:?}, pop_back = {last:?}, left = {q:?}");


The deque still indexes like a `Vec` (`q[0]` is the logical front, wherever it sits in the ring), converts from and into `Vec` for free, and rotates: `rotate_left(n)` walks every element toward the front by `n` positions, wrapping — a cheap "oldest first" reshuffle for ring-shaped data.


In [ ]:
// 8.2 Conversion, indexing, rotation, and the two slices.
let mut q = VecDeque::from(vec![10, 20, 30, 40]);
q.push_front(5);
println!("q[0] = {}, q[2] = {}", q[0], q[2]);      // logical indexing
q.rotate_left(2);
println!("rotated = {q:?}");
println!("as_slices = {:?}", q.as_slices());       // the ring, unrolled
let back_to_vec: Vec<i32> = Vec::from(q);
println!("as Vec = {back_to_vec:?}");


## 9. Iterating Collections

Every container in this chapter hands out its iterator through the **same three doors**, and the door you pick decides what happens to the collection. The distinction is chapter 05's borrow rules wearing an API:

| Method | Yields | Collection afterwards |
|---|---|---|
| `iter()` | `&T` (or `(&K, &V)`) | untouched |
| `iter_mut()` | `&mut T` | elements editable in place |
| `into_iter()` | `T` (or `(K, V)`) | consumed, dead |

`for x in &v` is sugar for `v.iter()`; `for x in &mut v` for `iter_mut()`; `for x in v` **moves the collection into the loop**, and using it after is `E0382`. For maps, the item is a `(&K, &V)` pair — you never get owned keys out of a borrowed iteration.


In [ ]:
// 9.1 The three doors, side by side.
let mut v = vec![10, 20, 30];

for x in &v { print!("{x} "); }                 // iter(): shared borrows
println!("(len {})", v.len());

for x in &mut v { *x *= 10; }                   // iter_mut(): editable in place
println!("after iter_mut: {v:?}");

let owned: Vec<i32> = v.into_iter().collect();  // into_iter(): consumed
println!("consumed into {owned:?}");
// println!("{v}");
// error[E0382]: borrow of moved value: `v` -- value moved here, in previous iteration


### Adapt and consume

Iteration is where chapter 03's closure introduction pays off. **Adaptors** (`map`, `filter`, `take`, `enumerate`) are lazy — they build a new iterator and do nothing until consumed; **consumers** (`sum`, `collect`, `max`, `count`, `any`) drive it. The full machinery is chapter 12's subject; here is the working set that collections demand daily:


In [ ]:
// 9.2 The daily driver set, one line each.
let prices = vec![101.5, 99.0, 105.25, 98.0];
let doubled: Vec<f64> = prices.iter().map(|p| p * 2.0).collect();
let highs: Vec<&f64> = prices.iter().filter(|p| **p >= 100.0).collect();
let total: f64 = prices.iter().sum();
let best = prices.iter().cloned().max_by(f64::total_cmp);   // f64 has no Ord
let any_under_98 = prices.iter().any(|p| *p < 98.0);
println!("doubled = {doubled:?}");
println!("highs = {highs:?}, total = {total}, best = {best:?}, any < 98: {any_under_98}");


In [ ]:
// 9.3 Collecting into shapes: BTreeSet, HashMap, and one owned String list.
let words = vec!["alpha", "beta", "alpha", "gamma"];
let unique: BTreeSet<&str> = words.iter().copied().collect();   // dedup, sorted
let by_key: HashMap<&str, i32> = [("a", 1), ("b", 2)].into_iter().collect();
let owned: Vec<String> = words.iter().map(|w| w.to_ascii_uppercase()).collect();
println!("{unique:?}");
println!("{by_key:?}");
println!("{owned:?}");


`collect()` type-checks against whatever container you name; where several are plausible, the binding's annotation decides (the turbofish `::<>` is the same choice written inline — the syntax chapter 01 previewed). One honest cost note: `filter` yields borrows, so a `Vec<&f64>` result carries references into the source — to own the elements, add `.cloned()`/`.copied()` before `collect`, or start the chain from `into_iter()`. Chapter 12 makes laziness and allocation behaviour precise; here, know which door you opened.


## 10. Collection Patterns You Will Actually Write

Four patterns cover most day-to-day container code. They are the reason the APIs above exist as they do.

### Grouping: fold a stream into a map of vectors

The count-with-`entry` pattern generalises: instead of a counter, the default is an empty `Vec` and every row pushes into its group. One pass, no lookups repeated, no `defaultdict` needed.


In [ ]:
// 10.1 Grouping rows by key with entry + or_insert_with.
let rows = [("a", 1.0), ("b", 2.0), ("a", 3.0), ("c", 4.0), ("b", 5.0)];
let mut groups: HashMap<&str, Vec<f64>> = HashMap::new();
for (key, value) in rows {
    groups.entry(key).or_insert_with(Vec::new).push(value);
}
for (key, values) in &groups {
    println!("{key}: {values:?} (sum {:.1})", values.iter().sum::<f64>());
}


### Sorting: `sort` in place, `sorted` via iterator

`sort()` (in place, needs `mut`) and `sort_unstable()` (faster, reorders equal elements) operate on slices, so both `Vec`s and arrays have them. `sort_by_key` extracts the comparison key. For a sorted *copy* without touching the original, collect through the iterator's `sorted()` — but note that takes `self`, so it fits an owned chain.


In [ ]:
// 10.2 The three sorting entry points.
let mut scores = vec![71, 95, 62, 88];
scores.sort();                          // ascending, in place
let descending: Vec<i32> = { let mut s = scores.clone(); s.sort(); s.reverse(); s };
let mut by_last_digit = scores.clone();
by_last_digit.sort_by_key(|s| s % 10);
println!("sorted = {scores:?}");
println!("descending = {descending:?}");
println!("by last digit = {by_last_digit:?}");


### Searching: `contains`, `position`, `binary_search`

`contains` scans linearly (O(n)); `iter().position(pred)` returns the index as `Option<usize>`; `binary_search` needs sorted data and answers in O(log n), returning `Ok(index)` on a hit and `Err(insertion_point)` on a miss — an unusual API that turns "not found" into "where it would go".


In [ ]:
// 10.3 The search inventory, including binary_search's Ok/Err duality.
let sorted = [10, 20, 30, 40, 50];
println!("contains 30: {}", sorted.contains(&30));
let words = vec!["alpha", "beta", "gamma"];
println!("position of beta: {:?}", words.iter().position(|w| *w == "beta"));
println!("binary_search 40: {:?}", sorted.binary_search(&40));
println!("binary_search 35: {:?}", sorted.binary_search(&35));  // Err = insertion point


### Deduplication: `sort` + `dedup`, or a set

`dedup()` removes *consecutive* duplicates — so it needs sorted (or grouped) input to do a whole-collection job. The set types do it in one insert each, unordered; sort-and-dedup keeps order and the original element type. Choose by which invariant you actually need.


In [ ]:
// 10.4 Two routes to uniqueness.
let mut data = vec![3, 1, 3, 2, 1, 3];
data.sort();
data.dedup();
println!("sorted + dedup: {data:?}");

let mut seen = std::collections::HashSet::new();
let unique_in_order: Vec<i32> = vec![3, 1, 3, 2, 1, 3]
    .into_iter()
    .filter(|x| seen.insert(*x))
    .collect();
println!("first-seen order: {unique_in_order:?}");


## 11. Ownership Inside Containers

The container is the owner; every element inside it is either moved in (non-`Copy`) or copied in (`Copy`). This section re-runs chapter 05's rules with a container in the middle, because the failure modes are the daily ones.

### Moving elements in and out


In [ ]:
// 11.1 push moves; pop/remove/drain move back out.
let mut owned_text: Vec<String> = Vec::new();
let title = String::from("AAPL");
owned_text.push(title);                 // move: `title` is dead here
let reclaimed: Option<String> = owned_text.pop();
println!("reclaimed = {reclaimed:?}, len = {}", owned_text.len());

let mut keep = vec![String::from("A"), String::from("B")];
let second = keep.swap_remove(1);       // O(1) removal: swaps last into the gap
println!("swap_remove took {second:?}, left {keep:?}");


`swap_remove` deserves the note: it deletes in O(1) by moving the *last* element into the removed slot — order-breaking, which is exactly why it is cheap. `remove(i)` preserves order and pays O(n) for it.

### The mutating loop: the classic E0502

You cannot iterate a collection while mutating it — iteration holds a shared borrow, mutation needs a mutable one, and the borrow rules say no. This is not a limitation to work around; it is the guarantee that no push ever invalidates the loop's internal pointer mid-iteration.


In [ ]:
// 11.2 The rejected loop, and the borrow-clean alternatives.
let mut numbers = vec![1, 2, 3];
// for n in &numbers { numbers.push(*n); }
// error[E0502]: cannot borrow `numbers` as mutable because it is also borrowed as immutable

// The fixes are semantic choices:
let doubled: Vec<i32> = numbers.iter().map(|n| n * 2).collect();   // new container
let count = numbers.len();
for _ in 0..count { numbers.push(0); }  // borrow ends before the mutation begins
numbers.retain(|n| *n != 0);            // in-place removal through &mut methods
println!("doubled = {doubled:?}, pruned = {numbers:?}");


### Copy elements escape the rule

`Vec<i32>` never fights you the way `Vec<String>` does: integers are `Copy`, so `v[0]` hands you a value, closures capture copies, and nothing is ever "moved out" in a way that kills the vector. The moment the element type owns heap data (`String`, `Vec`, `Box`), every access becomes a borrow-or-move decision — which is precisely why §2's deref-to-slice signature rule matters most for `Vec<String>`-shaped data.


In [ ]:
// 11.3 Copy vs non-Copy elements under iteration.
let ints = vec![1, 2, 3];
let doubled: Vec<i32> = ints.iter().map(|n| n * 2).collect();
println!("ints still alive: {ints:?}, doubled = {doubled:?}");

let texts = vec![String::from("a"), String::from("bb")];
let lengths: Vec<usize> = texts.iter().map(|t| t.len()).collect();  // borrows only
println!("lengths = {lengths:?}, texts untouched: {texts:?}");
// texts.iter().map(|t| *t).collect::<Vec<String>>() would be E0507:
// cannot move out of `*t` which is behind a shared reference


## 12. Reading the Compiler: Panics and E-Codes

Containers concentrate three failure families: runtime panics from bad access, trait-bound errors from the wrong key or wrong format, and the ownership diagnostics chapter 05 catalogued, now wearing element types. Probe-verified on this toolchain (rustc 1.98.1, debug build):

| Trigger | What you get |
|---|---|
| `v[i]` past the end | panic: `index out of bounds: the len is 2 but the index is 9` |
| `m[k]` with absent key | panic: `no entry found for key` |
| `s[i]` on a `String` | E0277: `String` cannot be indexed by `{integer}` |
| `println!("{}", set)` | E0277: doesn't implement `std::fmt::Display` (use `{:?}`) |
| `set + 1`, `vec + 1` | E0369: cannot add `{integer}` to `HashSet<i32>` |
| `m.insert(f, v)` | E0277/E0308: key type must be `Eq` + `Hash` |


In [ ]:
// 12.1 The panic family, caught, with their payloads.
let names = vec!["a", "b"];
let oob = std::panic::catch_unwind(|| names[9]);
println!("index panic payload: {:?}", oob.unwrap_err().downcast_ref::<String>());

let empty: HashMap<&str, i32> = HashMap::new();
let missing = std::panic::catch_unwind(|| empty["x"]);
println!("missing-key panic payload: {:?}", missing.unwrap_err().downcast_ref::<String>());

let gone: Option<i32> = Vec::<i32>::new().pop();
let _ = gone.unwrap_or_default();       // Option::unwrap panics with a &str payload,
                                        // unlike the two above, which format a String


Three payload shapes in one section: the index and map panics **format a `String`** at panic time (the length and index are baked into the message); `Option::unwrap` panics with a **static `&str`** (`called \`Option::unwrap()\` on a \`None\` value`) and costs nothing until it fires. `get`/`first`/`last` return `Option` for exactly this reason — absence becomes a value you match on instead of a message you crash into.

The E-codes carry their standard reading. E0277 is the trait-bound error: the container (or its formatter) demanded a trait the type does not implement, and the fix is usually `{:?}` or a different key type. E0369 is operator absence: collections implement `Add` only for `Vec` + `Vec` concatenation, never collection-plus-scalar. E0382/E0502 keep their chapter 05 meanings, now triggered by pushes, pops and iteration — §11's material.


In [ ]:
// 12.2 Compile errors are comments: the four container-shaped ones.
// let s = String::from("abc");
// let c = s[0];
// error[E0277]: the type `str` cannot be indexed by `{integer}`
//   (chapter 04's boundary rule, resurfacing through the collection lens)

// let set: std::collections::HashSet<i32> = [1].into_iter().collect();
// println!("{}", set);
// error[E0277]: `HashSet<i32>` doesn't implement `std::fmt::Display`
//   = note: in format strings you may be able to use `{:?}` instead

// let _ = set + 1;
// error[E0369]: cannot add `{integer}` to `HashSet<i32>`

// let m: std::collections::HashMap<f64, i32> = HashMap::new();
// error-free to *build*, but inserting needs Eq+Hash: f64 is rejected at use


## 13. Performance Notes for Low-Latency Code

For the market-data path this course keeps returning to, container choices *are* latency decisions. The facts that matter, in order of impact:

| Decision | Effect |
|---|---|
| `with_capacity(n)` vs growing | removes reallocation spikes the average hides |
| `&[T]` parameters | zero-cost: no new pointer ever |
| `Vec` vs `HashMap` for small N | linear scan of 8–16 elements beats hashing |
| `swap_remove` vs `remove` | O(1) vs O(n) tail shift, at the price of order |
| `retain` vs collect-new | in place, no allocation |
| `String::push` vs `format!` | one buffer, no intermediate |


In [ ]:
// 13.1 The splice cost, measured honestly on both containers.
use std::time::Instant;

const N: i32 = 30_000;
let mut v = Vec::with_capacity(N as usize);
let t0 = Instant::now();
for i in 0..N { v.push(i); }                    // back only: amortised O(1)
println!("Vec push_back {N}: {:?}", t0.elapsed());

let mut w = Vec::with_capacity(N as usize);
let t1 = Instant::now();
for i in 0..N { w.insert(0, i); }               // front only: O(n) per insert
println!("Vec insert(0, ..) {N}: {:?}", t1.elapsed());

let mut q = std::collections::VecDeque::with_capacity(N as usize);
let t2 = Instant::now();
for i in 0..N { q.push_front(i); }              // the deque's whole point
println!("VecDeque push_front {N}: {:?}", t2.elapsed());


Run the cell twice — the first run includes allocator warm-up, and the deque line should land orders of magnitude under the `insert(0, ..)` line. The point is not the exact numbers (they move with hardware and load); it is that *same operation, different container* spans three orders of magnitude, and nothing warns you at the call site. Chapter 01's rule holds: measure, then choose.

One pointer-shaped note: a `Vec` is three words to move and one allocation to clone; a `HashMap` is a heavier structure with a randomized hasher built on first use. Neither cost is visible in O-notation, and both show up in a flamegraph.


## 14. Mental Model: Length, Capacity, Owner

Three numbers and one name describe every container in this chapter:

- **length** — how many elements exist right now;
- **capacity** — how many fit before the next reallocation;
- **owner** — the single binding responsible for freeing the buffer.

A `Vec` is a three-word struct { pointer, length, capacity } pointing at a heap buffer. Moving it copies three words; cloning it duplicates the buffer. A `String` is the same shape with a UTF-8 guarantee. `HashMap`/`HashSet` are hash tables with the same owner discipline. `VecDeque` is a ring over one allocation. `BTreeMap` is a tree of owned nodes.

| Question to ask | Where it leads |
|---|---|
| Who owns this container? | who may mutate it, and when it dies |
| Length or capacity — which bit? | `len()` vs `capacity()`, off-by-one realism |
| Will it be searched by key or scanned? | HashMap vs Vec vs BTreeMap |
| Does order matter? | BTree (sorted) vs HashMap (random) vs Vec (insertion) |


In [ ]:
// 14.1 Length, capacity, and the owner's lifecycle, observed.
let mut v = Vec::with_capacity(4);
println!("start: len {} cap {}", v.len(), v.capacity());
for i in 0..4 { v.push(i); }
println!("filled: len {} cap {} (no growth yet)", v.len(), v.capacity());
v.push(4);
println!("one more: len {} cap {} (grew)", v.len(), v.capacity());
v.clear();
println!("cleared: len {} cap {} (buffer kept)", v.len(), v.capacity());

let owner = vec![String::from("x")];
let moved = owner;                      // three words, not the buffer
println!("moved = {moved:?}");


The owner rule composes with `Drop`: when the owner leaves scope, the container drops its elements **and then** its buffer — deterministic destruction, no garbage collector, in reverse declaration order. There is no way to forget to free a `Vec` and no way to free it twice; the compiler proves both.


## 15. Habits and Review

Habits that keep container code boring — in the good sense:

- **Reach for `Vec` first.** Add `HashMap`/`HashSet` when a lookup-by-key shape actually appears, not preemptively.
- **Take `&[T]`, return owned.** The slice parameter accepts arrays, `Vec`s and subslices; the owned return hands the caller a value with no lifetime attached.
- **Index with `[]` only when absence is a bug.** Otherwise `get` and match.
- **`entry` for any get-or-insert.** One lookup, no special-cased first-iteration flag.
- **Prefer adaptors over index loops.** `iter().map(..).collect()` states intent; `for i in 0..v.len()` invites off-by-ones.
- **Name the capacity when you can.** `with_capacity` is free latency.
- **Sorted display is a `BTreeMap` need.** If you keep sorting a `HashMap`'s output every time, the data structure is telling you something.

### Quick self-check

Answers are all in this notebook — each one names its section.

1. What is the type and meaning of `v.capacity()` after `Vec::with_capacity(4)` and four pushes? (§14.1)
2. Why does `for x in v` compile but kill `v` for the rest of the scope? (§9.1)
3. Which two bounds must a `HashMap` key satisfy, and which everyday type fails one of them? (§5)
4. What does `b.is_subset(&a)` return, and what do `union(&b)`-style methods return before you collect them? (§6)
5. What are the two panics `v[i]` and `m[k]` can produce, and what do `get`-style APIs return instead? (§12.1)


## 16. Cheat Sheet

| I want to… | Do this |
|---|---|
| build a list | `vec![a, b]`, `Vec::new()` + `push` |
| reserve space | `Vec::with_capacity(n)` |
| read safely | `v.get(i)` → `Option<&T>`; `v[i]` if absence is a bug |
| pass any sequence | `fn f(x: &[T])`; call as `f(&vec)` / `f(&arr)` |
| count / group by key | `map.entry(k).or_insert(0) += 1` / `.or_insert_with(Vec::new).push(x)` |
| unique elements | `HashSet` + `insert`'s bool, or sort + `dedup` |
| set algebra | `a.union(&b).collect::<Vec<_>>()` (also `intersection`, `difference`, …) |
| sorted map / range scan | `BTreeMap`, `.range(a..b)`, `first_key_value`, `pop_last` |
| both-ends queue | `VecDeque`: `push_front` / `push_back` / `pop_front` / `pop_back` |
| iterate without consuming | `for x in &v` |
| mutate elements | `for x in &mut v` |
| consume into a new shape | `v.into_iter().map(..).collect::<Vec<_>>()` |
| remove by predicate | `v.retain(\|x\| keep(x))` |
| sort | `v.sort()`, `v.sort_by_key(\|x\| k(x))`, `sort_unstable` when order of equals is free |
| find | `contains`, `iter().position(p)`, `binary_search` on sorted data |
| format a container | always `{:?}` (`{:#?}` for pretty) |


### What Comes Next

Chapter 06 fills the toolbox: from here on, the notebooks assume you can pick and drive a container without ceremony.

| Chapter | Title | What it settles |
|---|---|---|
| **07** | **I/O and Files** | `std::fs`, readers and writers, command-line arguments — containers meeting the outside world |
| 08 | Assertions and Type Checks | `assert!`, `matches!`, `size_of` — checking the assumptions this chapter made |
| 10 | Structs, OOP and Traits | your own types in these containers, and the traits that make them work (`Display`, `Debug`, `Ord`, `Hash`) |

**Carry forward.** You should now be able to do these without looking anything up:

- build, index and grow a `Vec`, and say what `len` vs `capacity` mean
- count or group rows with `entry`, and explain why it beats double lookup
- choose between `HashMap`, `BTreeMap`, `HashSet` and `Vec` by the access pattern
- iterate the three ways and predict which one kills the collection
- write functions that take `&[T]` and accept arrays, `Vec`s and slices alike
- read the container panics and E-codes (E0277, E0369, E0502) and know their fixes

If any of those still needs looking up, §16 is the one-page answer, and re-running the matching section here is the check.
